# stage7-duck-concurrency — vLLM aggregate throughput vs. concurrency

**Free diagnostic kernel. Plays no games. Consumes no submission quota.**

Cells 1–5 below are **byte-identical** to the production submission notebook
(`lb-9-arc3-duck-v12-with-qwen-3-8-27b`), so vLLM boots the real
`Qwen3.8-27B-FP8` on the real `NvidiaRtxPro6000` with the real bundled server
arguments and attention backend. The final cell replaces the Duck harness's
game loop with a fixed generation workload issued at several concurrency
levels.

**Question.** A competition rerun has a fixed ~9h wall-clock. Aggregate
throughput is a property of the server, so total tokens generated ≈
`throughput × wall_clock` *regardless* of concurrency — raising concurrency
28 → 37 gives each game more wall-clock but a proportionally thinner slice of
the GPU, and tokens-per-game come out about the same. **Concurrency is only a
win if aggregate throughput actually rises with more concurrent sequences.**
A real run showed KV-cache utilisation at only ~22%, which *suggests* headroom
— but that is an inference, not a measurement. This kernel measures it.

**Caveat on prompt length.** The analyzer runs a 32K rolling window at steady
state; this benchmark uses ~11–13K-token prompts (unique per request, so
prefix caching cannot collapse the prefill) as a compromise between realism and
prefill cost. Absolute KV-utilisation figures here are therefore *lower* than
production's; the *shape* of the throughput-vs-concurrency curve is the result
of interest.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
# =============================================================================
# stage7-duck-concurrency: vLLM aggregate-throughput vs. concurrency benchmark
# =============================================================================
# This cell REPLACES the Duck harness's own `bm.run(...)` game loop. Cells 1-5
# above are byte-identical to the production submission notebook, so vLLM is
# booted with the real Qwen3.8-27B-FP8 model, on the real RTX PRO 6000, with
# the real bundled server arguments. Nothing about the model, the machine
# shape, or the attention backend is changed.
#
# The question being measured (see experiments/stage7_duck_concurrency.md):
#
#   Total wall-clock in a competition rerun is fixed (~9h) and aggregate
#   throughput is a property of the vLLM server, so total tokens generated is
#   roughly `throughput x wall_clock` REGARDLESS of concurrency. Raising
#   concurrency 28 -> 37 gives each game more wall-clock but a proportionally
#   thinner slice of the GPU. Concurrency is therefore only a win if AGGREGATE
#   throughput actually RISES with more concurrent sequences.
#
# So: hold the workload fixed, sweep concurrency, measure aggregate output
# tokens/sec. Everything else (KV-cache utilisation, queueing, preemption) is
# recorded to explain whatever the throughput curve does.

import json
import os
import random
import time
import urllib.request

import asyncio

CONCURRENCY_LEVELS = [14, 28, 37, 48, 64]
PROMPT_CHARS_TARGET = 42000  # ~11-13K tokens; representative of the analyzer's
# 32K rolling window at steady state (see notebook 0 markdown for the caveat)
MAX_OUTPUT_TOKENS = 512
WARMUP_CONCURRENCY = 4
WARMUP_OUTPUT_TOKENS = 32

_base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "").rstrip("/")
MODEL_ID = os.environ.get("INFERENCE_ANALYZER_MODEL") or ""
if _base.endswith("/v1"):
    CHAT_URL = f"{_base}/chat/completions"
    METRICS_URL = f"{_base[:-3].rstrip('/')}/metrics"
else:
    CHAT_URL = f"{_base}/v1/chat/completions"
    METRICS_URL = f"{_base}/metrics"

print("=" * 78, flush=True)
print("stage7-duck-concurrency :: vLLM throughput-vs-concurrency benchmark")
print("=" * 78, flush=True)
print(f"LOCAL_ANALYZER_BASE_URL = {_base!r}")
print(f"INFERENCE_ANALYZER_MODEL = {MODEL_ID!r}")
print(f"chat url    = {CHAT_URL}")
print(f"metrics url = {METRICS_URL}")
print(f"levels      = {CONCURRENCY_LEVELS}")
print(f"max_tokens  = {MAX_OUTPUT_TOKENS}", flush=True)


# --- what the bundled setup actually launched vLLM with ----------------------
# The server's own --max-num-seqs is the single most decisive number for this
# question: if it sits below a swept level, that level cannot actually run
# that many sequences concurrently no matter what the client does.
def _dump_vllm_launch_args() -> None:
    print("\n--- bundled vLLM launch arguments (grepped from setup_commands.json) ---")
    path = BUNDLE_DIR / "setup_commands.json"  # noqa: F821  (defined in cell 3)
    try:
        commands = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:  # pragma: no cover - diagnostic only
        print(f"  could not read {path}: {exc!r}")
        return
    needles = (
        "max-num-seqs",
        "max_num_seqs",
        "max-model-len",
        "max_model_len",
        "gpu-memory-utilization",
        "gpu_memory_utilization",
        "max-num-batched-tokens",
        "max_num_batched_tokens",
        "enable-prefix-caching",
        "enable_prefix_caching",
        "tensor-parallel",
        "VLLM_ATTENTION_BACKEND",
        "VLLM_USE_FLASHINFER_SAMPLER",
    )
    hits = 0
    for command in commands:
        for line in str(command).splitlines():
            if any(n in line for n in needles):
                print(f"  {line.strip()[:200]}")
                hits += 1
    if not hits:
        print("  (no matching lines -- vLLM args are probably defaults)")


_dump_vllm_launch_args()


# --- server-side metrics -----------------------------------------------------
_METRIC_KEYS = (
    "vllm:gpu_cache_usage_perc",
    "vllm:num_requests_running",
    "vllm:num_requests_waiting",
    "vllm:num_preemptions_total",
    "vllm:gpu_prefix_cache_hit_rate",
)


def _scrape_metrics() -> dict[str, float]:
    """Scrape the vLLM Prometheus endpoint. Returns {} if unavailable."""
    out: dict[str, float] = {}
    try:
        with urllib.request.urlopen(METRICS_URL, timeout=5) as response:
            body = response.read().decode("utf-8", "ignore")
    except Exception:
        return out
    for line in body.splitlines():
        if line.startswith("#"):
            continue
        for key in _METRIC_KEYS:
            if line.startswith(key):
                try:
                    out[key] = max(out.get(key, float("-inf")), float(line.rsplit(" ", 1)[1]))
                except (ValueError, IndexError):
                    pass
    return out


# --- workload ----------------------------------------------------------------
_WORD_POOL = (
    "grid cell frame segment component adjacency containment transition action "
    "reset click hypothesis observe predict verify colour region boundary shape "
    "object move rotate reflect fill count index level score attempt policy "
    "state node edge path search branch prune candidate evidence contradiction"
).split()


def _make_prompt(seed: int) -> str:
    """A unique, realistically-long prompt.

    Deliberately UNIQUE per request (seeded RNG) so vLLM's prefix cache cannot
    collapse the prefill across the concurrent batch -- in the real harness
    every game holds its own independent conversation, so a shared-prefix
    workload would flatter the server in a way the real run never sees.
    """
    rng = random.Random(seed * 7919 + 13)
    parts = [
        f"Session {seed}. You are analysing an unfamiliar grid puzzle environment.\n",
        "Below is a transcript of observations, segmentations and attempted actions.\n\n",
    ]
    total = sum(len(p) for p in parts)
    step = 0
    while total < PROMPT_CHARS_TARGET:
        step += 1
        words = " ".join(rng.choice(_WORD_POOL) for _ in range(rng.randint(14, 30)))
        chunk = (
            f"[step {step}] observation: {words}. "
            f"segmentation: {rng.randint(2, 40)} components, "
            f"largest={rng.randint(3, 900)} px at ({rng.randint(0, 63)},{rng.randint(0, 63)}). "
            f"action taken: ACTION{rng.randint(1, 7)} -> "
            f"{'frame changed' if rng.random() < 0.5 else 'no change'}.\n"
        )
        parts.append(chunk)
        total += len(chunk)
    parts.append(
        "\nWrite a detailed step-by-step analysis of what mechanic this environment "
        "most likely implements, and what you would try next. Be thorough and specific.\n"
    )
    return "".join(parts)


async def _one_request(session, seed: int, max_tokens: int, allow_ignore_eos: bool):
    """Stream one completion; return per-token arrival timestamps."""
    payload = {
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": _make_prompt(seed)}],
        "max_tokens": max_tokens,
        "temperature": 0.8,
        "top_p": 0.95,
        "stream": True,
        "stream_options": {"include_usage": True},
    }
    if allow_ignore_eos:
        # Forces every sequence to generate exactly max_tokens, so the batch is
        # a fixed, identical workload at every concurrency level rather than
        # one whose size depends on when the model happens to stop.
        payload["ignore_eos"] = True

    t_send = time.perf_counter()
    arrivals: list[float] = []
    usage = None
    status = None
    error = None
    try:
        async with session.post(CHAT_URL, json=payload) as response:
            status = response.status
            if status != 200:
                error = (await response.text())[:400]
                return {
                    "ok": False, "status": status, "error": error, "seed": seed,
                    "t_send": t_send, "t_end": time.perf_counter(), "arrivals": [],
                    "usage": None,
                }
            async for raw in response.content:
                line = raw.decode("utf-8", "ignore").strip()
                if not line.startswith("data:"):
                    continue
                data = line[5:].strip()
                if data == "[DONE]":
                    break
                try:
                    obj = json.loads(data)
                except json.JSONDecodeError:
                    continue
                if obj.get("usage"):
                    usage = obj["usage"]
                for choice in obj.get("choices") or []:
                    delta = choice.get("delta") or {}
                    if delta.get("content"):
                        arrivals.append(time.perf_counter())
    except Exception as exc:  # pragma: no cover - diagnostic only
        error = repr(exc)[:400]
    return {
        "ok": error is None and bool(arrivals),
        "status": status,
        "error": error,
        "seed": seed,
        "t_send": t_send,
        "t_end": time.perf_counter(),
        "arrivals": arrivals,
        "usage": usage,
    }


async def _sampler(stop_event, samples: list[dict]):
    """Poll /metrics once a second for the duration of a level."""
    while not stop_event.is_set():
        metrics = _scrape_metrics()
        if metrics:
            samples.append(metrics)
        try:
            await asyncio.wait_for(stop_event.wait(), timeout=1.0)
        except asyncio.TimeoutError:
            pass


async def _run_level(session, concurrency: int, max_tokens: int, seed_base: int,
                     allow_ignore_eos: bool) -> dict:
    """Fire `concurrency` simultaneous requests and measure the batch."""
    samples: list[dict] = []
    stop_event = asyncio.Event()
    sampler_task = asyncio.create_task(_sampler(stop_event, samples))

    t0 = time.perf_counter()
    results = await asyncio.gather(*[
        _one_request(session, seed_base + i, max_tokens, allow_ignore_eos)
        for i in range(concurrency)
    ])
    t1 = time.perf_counter()
    stop_event.set()
    await sampler_task

    ok = [r for r in results if r["ok"]]
    failed = [r for r in results if not r["ok"]]

    # Token accounting: prefer the server's own usage numbers; fall back to
    # counting streamed content deltas (1 delta ~= 1 token in vLLM).
    usage_out = sum((r["usage"] or {}).get("completion_tokens", 0) for r in ok)
    delta_out = sum(len(r["arrivals"]) for r in ok)
    total_out = usage_out or delta_out
    prompt_toks = [
        (r["usage"] or {}).get("prompt_tokens", 0) for r in ok if r["usage"]
    ]

    # END-TO-END: what the harness actually experiences -- includes prefill,
    # queueing and ramp. This is the number that governs "tokens generated in
    # a fixed 9h wall-clock".
    e2e_wall = t1 - t0
    e2e_tps = total_out / e2e_wall if e2e_wall > 0 else 0.0

    # STEADY DECODE: the window in which ALL `concurrency` sequences are
    # simultaneously decoding -- from the last request's first token to the
    # first request's last token. Isolates decode from prefill/ramp.
    decode_tps = float("nan")
    decode_window = float("nan")
    decode_tokens = 0
    firsts = [r["arrivals"][0] for r in ok if r["arrivals"]]
    lasts = [r["arrivals"][-1] for r in ok if r["arrivals"]]
    if firsts and lasts:
        w_start, w_end = max(firsts), min(lasts)
        if w_end > w_start:
            decode_window = w_end - w_start
            for r in ok:
                decode_tokens += sum(1 for t in r["arrivals"] if w_start <= t <= w_end)
            decode_tps = decode_tokens / decode_window

    ttfts = [r["arrivals"][0] - r["t_send"] for r in ok if r["arrivals"]]

    def _agg(key, fn=max):
        vals = [s[key] for s in samples if key in s]
        return fn(vals) if vals else float("nan")

    return {
        "concurrency": concurrency,
        "requests_ok": len(ok),
        "requests_failed": len(failed),
        "first_error": (failed[0]["error"] if failed else None),
        "e2e_wall_s": e2e_wall,
        "e2e_out_tokens": total_out,
        "e2e_agg_tps": e2e_tps,
        "e2e_per_seq_tps": e2e_tps / concurrency if concurrency else 0.0,
        "decode_window_s": decode_window,
        "decode_tokens": decode_tokens,
        "decode_agg_tps": decode_tps,
        "decode_per_seq_tps": decode_tps / concurrency if concurrency else float("nan"),
        "ttft_p50_s": sorted(ttfts)[len(ttfts) // 2] if ttfts else float("nan"),
        "ttft_max_s": max(ttfts) if ttfts else float("nan"),
        "prompt_tokens_mean": (sum(prompt_toks) / len(prompt_toks)) if prompt_toks else 0,
        "kv_cache_usage_max": _agg("vllm:gpu_cache_usage_perc"),
        "running_max": _agg("vllm:num_requests_running"),
        "waiting_max": _agg("vllm:num_requests_waiting"),
        "preemptions_total_end": _agg("vllm:num_preemptions_total"),
        "metric_samples": len(samples),
    }


async def _main() -> list[dict]:
    try:
        import aiohttp
    except ImportError:
        print("aiohttp unavailable -- cannot run the benchmark.", flush=True)
        return []

    timeout = aiohttp.ClientTimeout(total=1800)
    connector = aiohttp.TCPConnector(limit=0)
    rows: list[dict] = []
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        # Probe once to find out whether the server accepts `ignore_eos`.
        probe = await _one_request(session, 999_000, 8, True)
        allow_ignore_eos = probe["ok"]
        if not allow_ignore_eos:
            print(f"note: ignore_eos probe failed (status={probe['status']}, "
                  f"err={probe['error']!r}); retrying without it", flush=True)
            probe = await _one_request(session, 999_001, 8, False)
            if not probe["ok"]:
                print(f"FATAL: server not answering: status={probe['status']} "
                      f"err={probe['error']!r}", flush=True)
                return []
        print(f"\nignore_eos accepted = {allow_ignore_eos}", flush=True)

        print(f"warmup: {WARMUP_CONCURRENCY} x {WARMUP_OUTPUT_TOKENS} tokens ...", flush=True)
        await _run_level(session, WARMUP_CONCURRENCY, WARMUP_OUTPUT_TOKENS,
                         900_000, allow_ignore_eos)
        print("warmup done.\n", flush=True)

        base_pre = _scrape_metrics()
        print(f"idle metrics: {base_pre}\n", flush=True)

        for i, level in enumerate(CONCURRENCY_LEVELS):
            print(f"--- level {level} (concurrency) starting ...", flush=True)
            row = await _run_level(session, level, MAX_OUTPUT_TOKENS,
                                   seed_base=1000 + i * 1000,
                                   allow_ignore_eos=allow_ignore_eos)
            rows.append(row)
            print(
                f"    conc={row['concurrency']:>3}  "
                f"ok={row['requests_ok']}/{row['concurrency']}  "
                f"e2e_agg={row['e2e_agg_tps']:.1f} tok/s  "
                f"decode_agg={row['decode_agg_tps']:.1f} tok/s  "
                f"per_seq={row['decode_per_seq_tps']:.2f} tok/s  "
                f"kv={row['kv_cache_usage_max']:.3f}  "
                f"waiting_max={row['waiting_max']}",
                flush=True,
            )
            # Let the scheduler drain fully so the next level starts clean.
            await asyncio.sleep(10)
    return rows


_rows = await _main()  # noqa: F704 - notebook top-level await, as in cell 9

# --- clearly-parseable result block ------------------------------------------
print("\n\n" + "=" * 78, flush=True)
print("BEGIN_CONCURRENCY_BENCHMARK_RESULTS")
print("=" * 78)
_hdr = (
    f"{'conc':>5} {'ok':>6} {'e2e_agg':>9} {'dec_agg':>9} {'per_seq':>8} "
    f"{'kv_max':>7} {'wait_max':>9} {'ttft_p50':>9} {'preempt':>8} {'prompt_tok':>11}"
)
print(_hdr)
print("-" * len(_hdr))
for _r in _rows:
    print(
        f"{_r['concurrency']:>5} "
        f"{_r['requests_ok']:>3}/{_r['concurrency']:<2} "
        f"{_r['e2e_agg_tps']:>9.1f} "
        f"{_r['decode_agg_tps']:>9.1f} "
        f"{_r['decode_per_seq_tps']:>8.2f} "
        f"{_r['kv_cache_usage_max']:>7.3f} "
        f"{_r['waiting_max']:>9.1f} "
        f"{_r['ttft_p50_s']:>9.2f} "
        f"{_r['preemptions_total_end']:>8.0f} "
        f"{_r['prompt_tokens_mean']:>11.0f}"
    )
print("-" * len(_hdr))
print("JSON_ROWS " + json.dumps(_rows))
print("=" * 78)
print("END_CONCURRENCY_BENCHMARK_RESULTS")
print("=" * 78, flush=True)

# --- the verdict arithmetic, computed in-kernel from the measured numbers ----
if len(_rows) >= 2:
    _by_conc = {r["concurrency"]: r for r in _rows}
    _r28, _r37 = _by_conc.get(28), _by_conc.get(37)
    print("\nVERDICT_INPUTS")
    if _r28 and _r37 and _r28["decode_agg_tps"] > 0:
        _ratio = _r37["decode_agg_tps"] / _r28["decode_agg_tps"]
        print(f"  aggregate decode throughput 37 / 28 = {_ratio:.4f}")
        print(f"  (>1.00 => raising concurrency generates MORE total tokens in a")
        print(f"   fixed 9h wall-clock; ~1.00 => token-neutral, only redistributes;")
        print(f"   <1.00 => raising concurrency generates FEWER total tokens)")
        print(f"  per-game token share 28 -> 37: waves 4 -> 3, so per-game "
              f"wall-clock x {4/3:.3f}, GPU slice x {_ratio*28/37:.3f}, "
              f"net per-game tokens x {_ratio*(28/37)*(4/3):.3f}")
    _best = max(_rows, key=lambda r: (r["decode_agg_tps"] if r["decode_agg_tps"] == r["decode_agg_tps"] else -1))
    print(f"  peak aggregate decode throughput at concurrency={_best['concurrency']}: "
          f"{_best['decode_agg_tps']:.1f} tok/s")
print("\nbenchmark complete.", flush=True)
